<div style="
  background: linear-gradient(135deg, #ff9a9e, #fad0c4, #fbc2eb);
  padding: 28px;
  border-radius: 24px;
  text-align: center;
  color: #3a2c4a;
  box-shadow: 0 8px 20px rgba(0,0,0,0.12);
">
  <h1 style="font-size: 38px; margin-bottom: 8px;">☕ Café Queue Simulation</h1>
  <h2 style="font-size: 22px; margin-top: 0;">Discrete-Probability Modeling of Arrivals, Menu Choice & Service</h2>
  <p style="font-size: 17px;">Poisson arrivals, categorical menu selection, and geometric service times with a dynamic third server. Seed 9248 everywhere for full reproducibility.</p>
</div>

## Model at a glance

Time is slotted in 5-minute intervals indexed by  = 0, 1, 2, \dots$. In each slot $:

- Arrivals  \overset{\mathrm{iid}}{\sim} \mathrm{Poisson}(\lambda)$ with base $\lambda = 0.8$ per slot.
- Each arrival picks one of 7 menu items from a categorical distribution with probabilities $.
- Each customer needs a geometric number of service slots  \sim \mathrm{Geom}(p_i)$ on $\{1, 2, \dots\}$, with (T_i = k) = (1 - p_i)^{k-1} p_i$.
- Two servers are always active; a third server activates in slot $ when {t-1} - 2 \ge h$ with default threshold  = 4$.
- Each active server makes one Bernoulli($) attempt per slot on a head-of-line customer; success completes that service.

**Menu selection probabilities $ and per-slot service success probabilities $:**

| Item | $ | $ |
|---|---|---|
| Coffee | 0.25 | 0.40 |
| Cake | 0.15 | 0.35 |
| Smoothie | 0.15 | 0.30 |
| Shake | 0.10 | 0.35 |
| Sandwich | 0.10 | 0.25 |
| Tea | 0.15 | 0.35 |
| Ice Cream | 0.10 | 0.30 |


In [ ]:
SEED <- 9248
set.seed(SEED)

library(ggplot2)
library(dplyr)
library(tidyr)
library(gridExtra)
library(knitr)

lambda_base <- 0.8
base_servers <- 2L
h_default <- 4L
slot_minutes <- 5

menu_q <- c(Coffee = 0.25, Cake = 0.15, Smoothie = 0.15, Shake = 0.10,
            Sandwich = 0.10, Tea = 0.15, Ice_Cream = 0.10)
menu_p <- c(Coffee = 0.40, Cake = 0.35, Smoothie = 0.30, Shake = 0.35,
            Sandwich = 0.25, Tea = 0.35, Ice_Cream = 0.30)
menu_names <- names(menu_q)

dir.create("results", showWarnings = FALSE)
kable(data.frame(Item = menu_names, q = as.numeric(menu_q), p = as.numeric(menu_p)),
      caption = "Menu selection probabilities q and per-attempt success probabilities p")


<div style="
  background: linear-gradient(135deg, #74c69d, #48cae4);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🚬 Warm-Up: Banach Matchbox</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Markov chain view, corrected PMF, and simulation check</p>
</div>

### The chain

Two boxes start full: $S_0 = (n, n)$. After $t$ draws the state is $S_t = (x_1, x_2)$, the matches still left in each box. Each step picks box 1 or box 2 with probability $1/2$ and removes one match:

$$S_{t+1} = \begin{cases} (x_1 - 1, x_2) & \text{with prob. } 1/2, \\ (x_1, x_2 - 1) & \text{with prob. } 1/2. \end{cases}$$

Only the current pair matters for the next step, so $P(S_{t+1} \mid S_t, \dots, S_0) = P(S_{t+1} \mid S_t)$: a discrete-time Markov chain on the $(x_1, x_2)$ grid. The walk stops at the first hitting time $\tau = \min\{t: x_1 = 0 \text{ or } x_2 = 0\}$, and we record $K = \max(x_1, x_2)$ at time $\tau$.

### Corrected distribution of $K$

Suppose box 1 is the one that empties first with $K = k \ge 1$ left in box 2. Then box 1 was picked exactly $n$ times, box 2 exactly $n - k$ times, and the final pick must be box 1. The first $2n - k - 1$ picks can be arranged freely:

$$P(\text{box 1 empties first}, K = k) = \binom{2n - k - 1}{n - 1} \left(\tfrac{1}{2}\right)^{2n - k}.$$

By symmetry the same holds for box 2, so for $k = 1, \dots, n$:

$$P(K = k) = 2\binom{2n - k - 1}{n - 1} \left(\tfrac{1}{2}\right)^{2n - k} = \binom{2n - k - 1}{n - 1} \left(\tfrac{1}{2}\right)^{2n - k - 1}.$$

Crucially, **$P(K = 0) = 0$** under this stopping rule: reaching $(0, 0)$ needs $2n$ draws, but one box is already empty after at most $2n - 1$ draws, so the walk always stops earlier. A formula such as $\binom{2n-k}{n}/2^{2n-k}$ assigns $P(K=0) \approx 0.176$ for $n = 10$ and halves the $k = n$ mass; it belongs to a different variant (empty box *discovered on the next attempt*) and must not be used here.

### Simulation check

The code below evaluates the corrected PMF for $n = 10$, simulates $100{,}000$ walks with seed 9248, tabulates the empirical distribution, and overlays theory against simulation. Both the figure and the comparison table are saved under `results/`.


In [ ]:
set.seed(9248)

banach_pmf <- function(n) {
  k <- 1:n
  p <- 2 * choose(2 * n - k - 1, n - 1) / 2^(2 * n - k)
  data.frame(k = k, analytical = p)
}

simulate_banach_once <- function(n) {
  box1 <- n
  box2 <- n
  while (box1 > 0 && box2 > 0) {
    if (runif(1) < 0.5) box1 <- box1 - 1 else box2 <- box2 - 1
  }
  max(box1, box2)
}

n_banach <- 10L
N_banach <- 100000L
sim_draws <- replicate(N_banach, simulate_banach_once(n_banach))
sim_tab <- as.data.frame(table(factor(sim_draws, levels = 1:n_banach)))
colnames(sim_tab) <- c("k", "count")
sim_tab$k <- as.integer(as.character(sim_tab$k))
sim_tab$simulated <- sim_tab$count / sum(sim_tab$count)

banach_cmp <- merge(banach_pmf(n_banach), sim_tab[, c("k", "simulated")], by = "k")
print(banach_cmp, digits = 5, row.names = FALSE)
cat(sprintf("\nP(K = 0) is structurally 0 here; every walk stops with K >= 1 (min simulated K = %d).\n", min(sim_draws)))

p_banach <- ggplot(banach_cmp, aes(x = k)) +
  geom_col(aes(y = analytical), fill = "steelblue", alpha = 0.6) +
  geom_point(aes(y = simulated), color = "red", size = 2) +
  geom_line(aes(y = simulated), color = "red", linewidth = 0.8) +
  labs(title = "Banach matchbox (n = 10): corrected theory vs simulation",
       x = "Matches left in the other box (k)", y = "Probability") +
  theme_minimal()
print(p_banach)
ggsave("results/banach_comparison.png", p_banach, width = 7, height = 4.5, dpi = 150)
write.csv(banach_cmp, "results/banach_comparison.csv", row.names = FALSE)


<div style="
  background: linear-gradient(135deg, #e8b7e8, #ffb6f4);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">📦 Building Blocks</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Poisson arrivals, categorical menu, geometric service times</p>
</div>

### Arrivals, choices, and service times

**Poisson arrivals.** The number of arrivals in one slot is $Z \sim \mathrm{Poisson}(\lambda)$ with $P(Z = k) = e^{-\lambda} \lambda^k / k!$ on $k = 0, 1, \dots$. Mean and variance are both $\lambda$; with $\lambda = 0.8$, $E[Z] = \mathrm{Var}(Z) = 0.8$. Over an interval scaled by $t/5$ minutes the rate scales linearly, $\lambda_t = \lambda \cdot t / 5$.

**Categorical menu choice.** Each arrival independently picks item $i$ with probability $q_i$ ($\sum_i q_i = 1$). Over many arrivals the counts are multinomial; the per-customer pick is the categorical building block.

**Geometric service.** A customer who ordered an item with per-slot success $p_i$ finishes in $T_i \sim \mathrm{Geom}(p_i)$ slots, $P(T_i = k) = (1 - p_i)^{k-1} p_i$ for $k = 1, 2, \dots$, with $E[T_i] = 1/p_i$ and $\mathrm{Var}(T_i) = (1 - p_i)/p_i^2$. Memorylessness means a customer still in service is never \u2018closer\u2019 to finishing: every slot is a fresh Bernoulli($p_i$) trial. The geometric law is the $r = 1$ special case of the negative binomial (trials until the first success).

The code below tabulates the Poisson PMF/CDF at $\lambda = 0.8$, the categorical menu, and a geometric example at $p = 0.30$, then plots all three distributions.


In [ ]:
k_vals <- 0:12
pmf_Z <- dpois(k_vals, lambda_base)
cdf_Z <- ppois(k_vals, lambda_base)
poisson_table <- data.frame(k = k_vals, PMF = round(pmf_Z, 6), CDF = round(cdf_Z, 6))
kable(poisson_table, caption = "Poisson (lambda = 0.8) PMF and CDF")
cat(sprintf("Poisson(lambda = %.1f): E[Z] = %.1f, Var(Z) = %.1f\n\n", lambda_base, lambda_base, lambda_base))

kable(data.frame(Item = menu_names, Prob = as.numeric(menu_q)), caption = "Categorical menu PMF")

p_example <- 0.30
k_t <- 1:20
pmf_T_ex <- dgeom(k_t - 1, p_example)
cat(sprintf("Geometric(p = %.2f): E[T] = %.4f slots, Var(T) = %.4f slots^2\n", p_example, 1 / p_example, (1 - p_example) / p_example^2))

df_pois <- data.frame(k = k_vals, pmf = pmf_Z)
p1 <- ggplot(df_pois, aes(x = factor(k), y = pmf)) + geom_col(fill = "skyblue") +
  labs(title = "Poisson PMF (lambda = 0.8)", x = "Arrivals k", y = "P(Z = k)") + theme_minimal()
df_cdf <- data.frame(k = k_vals, cdf = cdf_Z)
p2 <- ggplot(df_cdf, aes(x = k, y = cdf)) + geom_step(color = "steelblue", linewidth = 1) + geom_point(color = "steelblue") +
  labs(title = "Poisson CDF (lambda = 0.8)", x = "k", y = "P(Z <= k)") + theme_minimal()
df_menu <- data.frame(Item = factor(menu_names, levels = menu_names), q = as.numeric(menu_q))
p3 <- ggplot(df_menu, aes(x = Item, y = q)) + geom_col(fill = "lightgreen", color = "grey30") +
  labs(title = "Categorical menu PMF", x = "", y = "Probability") + theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))
df_geo <- data.frame(k = k_t, pmf = pmf_T_ex)
p4 <- ggplot(df_geo, aes(x = factor(k), y = pmf)) + geom_col(fill = "pink", color = "grey30") +
  labs(title = "Geometric PMF (p = 0.30)", x = "Service slots k", y = "P(T = k)") + theme_minimal()
p_blocks <- grid.arrange(p1, p2, p3, p4, ncol = 2)
ggsave("results/building_blocks.png", p_blocks, width = 10, height = 7, dpi = 150)
write.csv(data.frame(k = k_vals, pmf = pmf_Z, cdf = cdf_Z), "results/poisson_pmf_cdf.csv", row.names = FALSE)
write.csv(data.frame(k = k_t, pmf = pmf_T_ex), "results/geometric_example_p030.csv", row.names = FALSE)


<div style="
  background: linear-gradient(135deg, #9668b1, #8fb5d7, #78b9f6);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🎲 Menu-Aware Sampling</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Empirical means, proportions, histograms, boxplots, ECDFs</p>
</div>

### From theory to simulated data

Theory predicts $E[Z] = \mathrm{Var}(Z) = 0.8$ arrivals per slot and menu shares exactly $q_i$. Drawing $N = 2000$ Poisson arrivals and $N = 2000$ categorical menu picks (seed 9248) checks that empirical means, variances, and proportions land near those values. Drawing $1000$ geometric service times per item shows why slow items clog the queue: Sandwich ($p = 0.25$) averages $4.0$ slots while Coffee ($p = 0.40$) averages $2.5$. Histograms, boxplots, and ECDFs make the spread visible, and rescaling $\lambda_t = \lambda \cdot t/5$ shows how longer observation windows shift the Poisson PMF.


In [ ]:
set.seed(9248)
N_samp <- 2000L
pois_sample <- rpois(N_samp, lambda_base)
cat(sprintf("Poisson sample (N = %d, lambda = %.1f): mean = %.4f, var = %.4f\n\n", N_samp, lambda_base, mean(pois_sample), var(pois_sample)))

items_sample <- sample(menu_names, size = N_samp, replace = TRUE, prob = menu_q)
item_counts <- table(factor(items_sample, levels = menu_names))
emp_menu <- data.frame(Item = names(item_counts), Count = as.integer(item_counts),
                       Proportion = round(as.numeric(item_counts) / N_samp, 4))
kable(emp_menu, caption = "Empirical menu shares (N = 2000, seed 9248)")
write.csv(emp_menu, "results/empirical_menu.csv", row.names = FALSE)

df_hist <- data.frame(Z = pois_sample)
p_overlay <- ggplot(df_hist, aes(x = Z)) +
  geom_histogram(aes(y = after_stat(density)), binwidth = 1, boundary = -0.5, fill = "lightblue", color = "grey30") +
  geom_point(data = data.frame(k = 0:max(6, max(pois_sample)), pmf = dpois(0:max(6, max(pois_sample)), lambda_base)),
             aes(x = k, y = pmf), color = "red", size = 2) +
  labs(title = "Simulated Poisson arrivals vs theory (N = 2000)", x = "Arrivals per slot", y = "Density") +
  theme_minimal()
print(p_overlay)
ggsave("results/poisson_sim_overlay.png", p_overlay, width = 7, height = 4.5, dpi = 150)


In [ ]:
set.seed(9248)
per_item_n <- 1000L
service_samples <- lapply(menu_p, function(pp) rgeom(per_item_n, prob = pp) + 1)
service_df <- data.frame(time = unlist(service_samples),
                         item = rep(names(menu_p), each = per_item_n))

p_hist <- ggplot(service_df, aes(x = time)) +
  geom_histogram(binwidth = 1, boundary = 0.5, fill = "skyblue", color = "grey30") +
  facet_wrap(~item, scales = "free_y") +
  labs(title = "Per-item service-time histograms (1000 draws each)", x = "Service time (slots)", y = "Count") +
  theme_minimal()
print(p_hist)
ggsave("results/service_histograms.png", p_hist, width = 10, height = 6, dpi = 150)

p_box <- ggplot(service_df, aes(x = item, y = time, fill = item)) +
  geom_boxplot() +
  labs(title = "Service-time spread by item", x = "", y = "Service time (slots)") +
  theme_minimal() + theme(axis.text.x = element_text(angle = 30, hjust = 1)) + guides(fill = "none")
print(p_box)
ggsave("results/service_boxplots.png", p_box, width = 8, height = 4.5, dpi = 150)

p_ecdf <- ggplot(dplyr::filter(service_df, item %in% c("Coffee", "Sandwich", "Ice_Cream")),
                 aes(x = time, color = item)) +
  stat_ecdf(linewidth = 1) +
  labs(title = "ECDFs: fast Coffee vs slow Sandwich", x = "Service time (slots)", y = "ECDF") +
  theme_minimal()
print(p_ecdf)
ggsave("results/service_ecdf.png", p_ecdf, width = 7, height = 4.5, dpi = 150)

slot_minutes_vec <- c(5, 10, 30)
lambda_vec <- lambda_base * (slot_minutes_vec / 5)
k_grid <- 0:15
pmf_list <- lapply(seq_along(slot_minutes_vec), function(i)
  data.frame(slot_min = slot_minutes_vec[i], k = k_grid, pmf = dpois(k_grid, lambda_vec[i])))
pmf_slots <- do.call(rbind, pmf_list)
p_slots <- ggplot(pmf_slots, aes(x = k, y = pmf, fill = factor(slot_min))) +
  geom_col(position = "dodge") +
  labs(title = "Poisson PMF rescales with window length", x = "Arrivals k", y = "P(Z = k)", fill = "Window (min)") +
  theme_minimal()
print(p_slots)
ggsave("results/slot_scaling.png", p_slots, width = 8, height = 4.5, dpi = 150)
write.csv(pmf_slots, "results/slot_scaling_pmf.csv", row.names = FALSE)


<div style="
  background: linear-gradient(135deg, #7dbcf3, #9ff98d, #d6a870);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">⏱️ Per-Item Service Analytics</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Expected service times and bottleneck identification</p>
</div>

### Who slows down the café?

With $E[T_i] = 1/p_i$ and $\mathrm{Var}(T_i) = (1 - p_i)/p_i^2$, Sandwich ($p = 0.25$) is the bottleneck at $4.0$ slots on average with variance $12.0$, followed by Smoothie and Ice Cream ($3.33$ slots, variance $7.78$). Coffee is fastest ($2.5$ slots, variance $3.75$). The mixture mean service time is $E[S] = \sum_i q_i / p_i \approx 3.0$ slots, so offered load per server at base $\lambda = 0.8$ is $\lambda E[S]/2 \approx 1.2 > 1$: the two-server café is nominally overloaded, which is why the queue only survives through the third server.


In [ ]:
menu_df <- data.frame(Item = names(menu_p), q = as.numeric(menu_q), p = as.numeric(menu_p))
menu_df$E_T <- 1 / menu_df$p
menu_df$Var_T <- (1 - menu_df$p) / (menu_df$p^2)
kable(menu_df, caption = "Per-item q, p, E[T], Var(T)", digits = 4)
cat(sprintf("Mixture E[S] = %.4f slots; offered load at lambda = %.1f on 2 servers: rho = %.3f\n",
            sum(menu_df$q * menu_df$E_T), lambda_base, lambda_base * sum(menu_df$q * menu_df$E_T) / 2))
write.csv(menu_df, "results/per_item_service.csv", row.names = FALSE)

p_et <- ggplot(menu_df, aes(x = reorder(Item, -E_T), y = E_T)) +
  geom_col(fill = "tan", color = "grey30") +
  geom_text(aes(label = sprintf("%.2f", E_T)), vjust = -0.4, size = 3.5) +
  labs(title = "Expected service time per item (slots)", x = "", y = "E[T] (slots)") +
  theme_minimal()
print(p_et)
ggsave("results/expected_service.png", p_et, width = 8, height = 4.5, dpi = 150)


<div style="
  background: linear-gradient(135deg, #f197a7, #efb66f, #f9fc95);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🏪 FIFO Café Queue Simulator</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">One FIFO discipline with dynamic third-server activation</p>
</div>

### One queue discipline, used everywhere

Each slot runs the same three steps. Arrivals $Z_t \sim \mathrm{Poisson}(\lambda)$ join the back, each with an independent categorical menu pick. The server count is set from the pre-service queue length $L_t$:

$$s_t = \begin{cases} 2, & L_t - 2 < h, \\ 3, & \text{otherwise,} \end{cases} \qquad h = 4 \text{ by default.}$$

Then the first $\min(s_t, L_t)$ head-of-line customers each attempt service once: customer $j$ departs with probability $p_{i(j)}$ and otherwise stays in place, preserving FIFO order. Failed customers are *not* rotated to the back; every later section reuses this exact function so load, $\lambda$-, $h$-, and joint experiments are directly comparable. The run records $Q_t$ (queue left after service, $Q_0 = 0$), the third-server flag, and the per-item composition for stacked-area plots.


In [ ]:
simulate_cafe <- function(lambda, num_slots = 2000L, menu_q_in = menu_q, menu_p_in = menu_p,
                          base_servers = 2L, h = 4L) {
  stopifnot(!is.null(names(menu_q_in)), identical(names(menu_q_in), names(menu_p_in)))
  item_names <- names(menu_q_in)
  Q <- integer(num_slots + 1L)
  queue <- character(0)
  queue_snap <- vector("list", num_slots + 1L)
  queue_snap[[1L]] <- character(0)
  third_active <- logical(num_slots)
  comp_mat <- matrix(0, nrow = num_slots + 1L, ncol = length(item_names))
  colnames(comp_mat) <- item_names
  for (t in seq_len(num_slots)) {
    arrivals <- rpois(1, lambda)
    if (arrivals > 0) queue <- c(queue, sample(item_names, arrivals, replace = TRUE, prob = menu_q_in))
    st <- if (length(queue) - base_servers >= h) 3L else 2L
    third_active[t] <- (st == 3L)
    if (length(queue) > 0) {
      serve_n <- min(st, length(queue))
      head <- queue[seq_len(serve_n)]
      tail <- if (serve_n < length(queue)) queue[(serve_n + 1):length(queue)] else character(0)
      done <- runif(serve_n) < as.numeric(menu_p_in[head])
      queue <- c(head[!done], tail)
    }
    Q[t + 1L] <- length(queue)
    queue_snap[[t + 1L]] <- queue
    if (length(queue) > 0) comp_mat[t + 1L, ] <- as.integer(table(factor(queue, levels = item_names)))
  }
  list(Q = Q, queue = queue_snap, comp = comp_mat, third_active = third_active,
       lambda = lambda, num_slots = num_slots, h = h)
}
summarize_run <- function(res) c(mean_Q = mean(res$Q), var_Q = var(res$Q),
                                 P_Q0 = mean(res$Q == 0), P_third = mean(res$third_active))


### Three load regimes, same simulator

Low ($\lambda = 0.4$), base ($\lambda = 0.8$), and high ($\lambda = 1.5$) loads each run $2000$ slots at $h = 4$ with seed 9248. For every run we report $E[Q]$, $\mathrm{Var}(Q)$, $P(Q = 0)$, and $P(\text{third active})$, plus the Q-series, the empirical PMF of $Q$ (long-run fraction of slots at each level), and the 0–400-slot item composition.


In [ ]:
set.seed(9248)
res_low <- simulate_cafe(lambda = 0.4, num_slots = 2000L, h = h_default)
res_base <- simulate_cafe(lambda = 0.8, num_slots = 2000L, h = h_default)
res_high <- simulate_cafe(lambda = 1.5, num_slots = 2000L, h = h_default)

scenario_table <- rbind(Low = summarize_run(res_low), Base = summarize_run(res_base), High = summarize_run(res_high))
print(round(scenario_table, 4))
write.csv(data.frame(scenario = rownames(scenario_table), round(scenario_table, 4), row.names = NULL),
          "results/load_scenarios.csv", row.names = FALSE)


In [ ]:
plot_Q_series <- function(res, title_label, color = "steelblue") {
  ggplot(data.frame(slot = 0:res$num_slots, Q = res$Q), aes(x = slot, y = Q)) +
    geom_line(color = color, linewidth = 0.5) +
    labs(title = title_label, x = "Slot", y = "Queue length Q") + theme_minimal()
}
plot_Q_pmf <- function(res, title_label, fill_color = "skyblue") {
  pmf <- as.data.frame(res$Q) |> setNames("Q") |> dplyr::count(Q) |> dplyr::mutate(prop = n / sum(n))
  ggplot(pmf, aes(x = Q, y = prop)) + geom_col(fill = fill_color, color = "grey30") +
    labs(title = title_label, x = "Queue length Q", y = "Fraction of slots") + theme_minimal()
}
plot_composition <- function(res, title_label) {
  comp_first <- as.data.frame(res$comp[1:401, , drop = FALSE])
  comp_first$slot <- 0:400
  comp_long <- tidyr::pivot_longer(comp_first, cols = -slot, names_to = "Item", values_to = "Count")
  ggplot(comp_long, aes(x = slot, y = Count, fill = Item)) + geom_area() +
    labs(title = title_label, x = "Slot", y = "Items waiting") + theme_minimal()
}

p_low <- plot_Q_series(res_low, "Queue over time: low load (lambda = 0.4)", "darkgreen")
p_low_pmf <- plot_Q_pmf(res_low, "Empirical PMF of Q: low load", "lightgreen")
p_base <- plot_Q_series(res_base, "Queue over time: base load (lambda = 0.8)")
p_base_pmf <- plot_Q_pmf(res_base, "Empirical PMF of Q: base load")
p_high <- plot_Q_series(res_high, "Queue over time: high load (lambda = 1.5)", "firebrick")
p_high_pmf <- plot_Q_pmf(res_high, "Empirical PMF of Q: high load", "salmon")
g_load <- grid.arrange(p_low, p_low_pmf, p_base, p_base_pmf, p_high, p_high_pmf, ncol = 2)
ggsave("results/load_series_pmf.png", g_load, width = 11, height = 9, dpi = 150)

p_low_comp <- plot_composition(res_low, "Queue mix 0-400: low load")
p_base_comp <- plot_composition(res_base, "Queue mix 0-400: base load")
p_high_comp <- plot_composition(res_high, "Queue mix 0-400: high load")
print(p_low_comp); print(p_base_comp); print(p_high_comp)
ggsave("results/load_comp_low.png", p_low_comp, width = 8, height = 4.5, dpi = 150)
ggsave("results/load_comp_base.png", p_base_comp, width = 8, height = 4.5, dpi = 150)
ggsave("results/load_comp_high.png", p_high_comp, width = 8, height = 4.5, dpi = 150)


<div style="
  background: linear-gradient(135deg, #74c69d, #b7e4c7, #48cae4);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">📈 Arrival-Rate Sensitivity</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">How E[Q], Var(Q), P(Q=0), and third-server use evolve with λ</p>
</div>

<div style="
  background: linear-gradient(135deg, #cdb4db, #ffc8dd, #ffafcc);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🎛️ Activation-Threshold Sensitivity</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Sweeping h at fixed λ = 0.8</p>
</div>

<div style="
  background: linear-gradient(135deg, #90be6d, #43aa8b, #577590);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🗺️ Joint λ–h Stability Map</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Where the café stays under control</p>
</div>

## Takeaways

(Filled in as each analysis section lands.)


In [ ]:
sessionInfo()
